# Phase 2 & 3 — Data Exploration, Manifest Building & Preprocessing Pipeline
## Heart Disease Detection using Phonocardiogram (PCG) and IoT
**BTech Major Project | Data Science | Galgotias College of Engineering & Technology**

**Authors:** Princee Singh (2300971630044) · Priyanshu Kumar (2300971630045)
**Supervisor:** Dr. Kalpna Sagar

---

### Objectives
**Phase 2 — Data Exploration & Manifest Building**
- Locate and load both datasets from the backed-up Drive zips
- Build a unified manifest CSV combining all clips from both datasets with
  consistent binary labels and group identifiers for leakage-safe splitting
- Perform exploratory listening to understand what the data sounds like

**Phase 3 — Preprocessing Pipeline**
- Implement a full signal cleaning pipeline: resample → bandpass filter →
  despike → wavelet denoise → trim silence → normalize amplitude
- Validate the pipeline by ear and by waveform inspection
- Run the pipeline across all 6,293 clips and cache processed signals as `.npy`

### Why These Two Phases Are in One Notebook
Phases 2 and 3 are tightly coupled — the manifest built in Phase 2 is
immediately used to drive the preprocessing batch in Phase 3. Keeping them
together avoids re-loading the manifest between sessions.


---
# PHASE 2 — Data Exploration & Manifest Building


## Step 1 — Mount Drive and Unzip Datasets

Datasets are unzipped to Colab's local disk (`/content/work`) rather than
accessed directly over the Drive mount. This is critical for performance —
reading thousands of small `.wav` files over a Drive-mounted network path
is extremely slow (10-100x slower than local SSD reads), and the preprocessing
pipeline in Phase 3 reads every single file.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/pcg-project'

!mkdir -p /content/work
!unzip -q "{PROJECT}/data/raw/physionet2016.zip" -d /content/work
!unzip -q "{PROJECT}/data/raw/circor2022.zip" -d /content/work
print("Unzip complete.")


Mounted at /content/drive
Unzip complete.


## Step 2 — Locate Label Files Recursively

**Why recursive search instead of hardcoded paths?**
When a folder is zipped using an absolute path (e.g. `zip -r out.zip /content/raw_dl/...`),
the full path is preserved inside the zip. Unzipping to `/content/work` creates
`/content/work/content/raw_dl/...` — a nested path that differs from the original.
Rather than hardcoding the expected path (which would break silently if the
nesting depth changed), we search recursively for the label files by name.
This makes the pipeline robust to any zip/unzip path variation.


In [ ]:
import pandas as pd, glob, os

ref_files = glob.glob('/content/work/**/REFERENCE.csv', recursive=True)
circor_csv_paths = glob.glob('/content/work/**/training_data.csv', recursive=True)

print(f"Found {len(ref_files)} REFERENCE.csv files (PhysioNet 2016)")
print(f"CirCor training_data.csv: {circor_csv_paths}")


Found 7 REFERENCE.csv files (PhysioNet 2016)
CirCor training_data.csv: ['/content/work/content/raw_dl/physionet.org/files/circor-heart-sound/1.0.3/training_data.csv']


### Finding
7 REFERENCE.csv files found — one per training subfolder (training-a through
training-f) plus one in the `validation/` subfolder. The validation subfolder
is the Challenge's original held-out test set and is excluded when building the
manifest (see Step 3).


## Step 3 — Build the PhysioNet 2016 Manifest

**Label encoding:** PhysioNet 2016 uses numeric labels — `1` for abnormal and
`-1` for normal — stored in each subfolder's `REFERENCE.csv`.

**Validation subfolder exclusion:** The `validation/` subfolder is explicitly
skipped. It was the Challenge organizers' held-out test set for the 2016
competition — pooling it with training data would inflate results and
misrepresent our model's real-world generalisability.


In [ ]:
rows = []
for ref_path in ref_files:
    if '/validation/' in ref_path:
        continue  # exclude Challenge's original held-out test set

    folder = os.path.dirname(ref_path)
    df = pd.read_csv(ref_path, header=None, names=["file", "label"])
    df["path"] = df["file"].apply(lambda f: os.path.join(folder, f"{f}.wav"))
    df["subset"] = os.path.basename(folder)
    rows.append(df)

manifest_2016 = pd.concat(rows, ignore_index=True)
manifest_2016["source"] = "physionet2016"
manifest_2016["label_mapped"] = manifest_2016["label"].map({1: "abnormal", -1: "normal"})

print(f"Shape: {manifest_2016.shape}")
print(manifest_2016["label_mapped"].value_counts())


Shape: (3541, 6)
label_mapped
normal      2725
abnormal     816
Name: count, dtype: int64


### Finding — PhysioNet 2016 Class Distribution
- **Total recordings:** 3,541 (across 6 training subfolders)
- **Normal:** 2,725 (76.9%)
- **Abnormal:** 816 (23.1%)
- **Class ratio:** ~3.3:1 (normal:abnormal)

This imbalance is intentional — it reflects realistic population-level
heart disease prevalence. It will be handled during training via a
class-weighted loss function (pos_weight in BCEWithLogitsLoss).


## Step 4 — Explore CirCor 2022 Structure

CirCor's data structure is fundamentally different from PhysioNet 2016:
- **One row per patient** (not per recording) in `training_data.csv`
- **Multiple recordings per patient** — one per auscultation site (AV/PV/TV/MV)
- **Patient-level label** in the `Murmur` column
- **Site-level detail** in `Murmur locations` — which specific sites the murmur was audible at

This means we cannot simply copy the patient label to every recording.
A patient labelled "Present" may only have an audible murmur at the AV site,
with clean recordings at PV/TV/MV. We must expand to per-recording labels
using the `Murmur locations` column.


In [ ]:
circor_csv = pd.read_csv(circor_csv_paths[0])
circor_base = os.path.dirname(circor_csv_paths[0])
wav_dir = os.path.join(circor_base, "training_data")

print(f"CirCor shape: {circor_csv.shape}")
print(f"Columns: {circor_csv.columns.tolist()}")
print()
print("Patient-level murmur distribution:")
print(circor_csv["Murmur"].value_counts())
print()
print(f"WAV directory exists: {os.path.exists(wav_dir)}")
print(f"WAV files found: {len(glob.glob(os.path.join(wav_dir, '*.wav')))}")


CirCor shape: (942, 23)
Columns: ['Patient ID', 'Recording locations:', 'Age', 'Sex', 'Height', 'Weight', 'Pregnancy status', 'Murmur', 'Murmur locations', 'Most audible location', 'Systolic murmur timing', 'Systolic murmur shape', 'Systolic murmur grading', 'Systolic murmur pitch', 'Systolic murmur quality', 'Diastolic murmur timing', 'Diastolic murmur shape', 'Diastolic murmur grading', 'Diastolic murmur pitch', 'Diastolic murmur quality', 'Outcome', 'Campaign', 'Additional ID']

Patient-level murmur distribution:
Murmur
Absent     695
Present    179
Unknown     68
Name: count, dtype: int64

WAV directory exists: True
WAV files found: 3163


## Step 5 — Build the CirCor 2022 Per-Recording Manifest

**First attempt — exact filename match:**
The first version of this code used an exact filename match
(`{patient_id}_{location}.wav`). This resulted in 43 unresolved files (~1.4%),
which on inspection turned out to be patients who had **multiple recordings at
the same site** — saved as `50782_AV_1.wav`, `50782_AV_2.wav` rather than a
plain `50782_AV.wav`. Exact matching missed these silently.

**Fix — prefix glob match:**
Switching to `glob(f"{patient_id}_{location}*.wav")` recovers all repeat
recordings automatically and also benefits training by providing additional
samples from those patients. 0 unresolved files after this fix.

**Label resolution logic:**
- Patient `Murmur = 'Absent'` → all their recordings labelled `absent`
- Patient `Murmur = 'Unknown'` → all their recordings labelled `unknown`
  (dropped in Step 6 — no reliable ground truth)
- Patient `Murmur = 'Present'` → recording labelled `present` only if the
  recording's site appears in `Murmur locations`; otherwise `absent`
  (the murmur wasn't audible at that particular site)


In [ ]:
rows = []
for _, r in circor_csv.iterrows():
    pid = r['Patient ID']
    locations = str(r['Recording locations:']).split('+')
    murmur_status = r['Murmur']
    murmur_locations = (
        str(r['Murmur locations']).split('+')
        if pd.notna(r['Murmur locations']) else []
    )

    for loc in locations:
        # prefix glob recovers repeat recordings (e.g. 50782_AV_1.wav, _2.wav)
        matches = glob.glob(os.path.join(wav_dir, f"{pid}_{loc}*.wav"))

        if murmur_status == 'Absent':
            label = 'absent'
        elif murmur_status == 'Unknown':
            label = 'unknown'
        else:  # Present
            label = 'present' if loc in murmur_locations else 'absent'

        for fpath in matches:
            rows.append({
                'file': os.path.basename(fpath), 'path': fpath,
                'patient_id': pid, 'location': loc,
                'label_mapped': label, 'source': 'circor2022'
            })

manifest_circor = pd.DataFrame(rows)

print(f"Shape: {manifest_circor.shape}")
print(manifest_circor['label_mapped'].value_counts())
print(f"All files exist: {manifest_circor['path'].apply(os.path.exists).all()}")


Shape: (3209, 6)
label_mapped
absent     2544
present     509
unknown     156
Name: count, dtype: int64
All files exist: True


### Finding — CirCor 2022 Per-Recording Distribution
- **Total recordings (after fix):** 3,209 (up from 3,120 with exact matching)
- **Absent:** 2,544 (79.3%)
- **Present:** 509 (15.9%)
- **Unknown:** 156 (4.9%) — will be dropped in Step 6
- **All files confirmed present on disk**

The 43 missing files from the first attempt were recovered by the prefix glob —
they were patients with multiple recordings at the same auscultation site.


## Step 6 — Merge into Unified Manifest

**Label mapping note (important for report):**
PhysioNet 2016 uses disease-level labels (normal/abnormal) while CirCor uses
symptom-level labels (murmur absent/present). These are related but not
strictly identical — a patient could have an abnormal heart sound that is not
a murmur, or a detectable murmur that is benign. We map
`abnormal ≈ murmur present` and `normal ≈ murmur absent`, which is the
standard approach in the literature (used by the majority of the 18 papers in
our literature review). This assumption is stated explicitly rather than left
as an implicit design choice.

**CirCor 'Unknown' rows dropped:** These 156 recordings represent cases where
even expert annotators could not confidently assign a label. Including them
with either label would introduce noise into both training and evaluation.


In [ ]:
final_2016 = manifest_2016[['path', 'label_mapped', 'source']].copy()
final_circor = (
    manifest_circor[manifest_circor['label_mapped'] != 'unknown']
    [['path', 'label_mapped', 'source']].copy()
)

unified_manifest = pd.concat([final_2016, final_circor], ignore_index=True)
unified_manifest['label_binary'] = unified_manifest['label_mapped'].map({
    'normal': 0, 'absent': 0, 'abnormal': 1, 'present': 1
})

# group_id for leakage-safe splitting in Phase 6
def get_group_id(path, source):
    fname = os.path.splitext(os.path.basename(path))[0]
    if source == 'circor2022':
        return fname.split('_')[0]   # CirCor patient ID (e.g. "50782")
    return fname                      # PhysioNet: each recording its own group

unified_manifest['group_id'] = unified_manifest.apply(
    lambda r: get_group_id(r['path'], r['source']), axis=1
)

print(f"Unified manifest shape: {unified_manifest.shape}")
print()
print("Per-source binary label counts:")
print(unified_manifest.groupby('source')['label_binary'].value_counts())
print()
print(f"Total unique groups (for split): {unified_manifest['group_id'].nunique()}")
print(f"Overall class balance: {unified_manifest['label_binary'].mean():.2%} abnormal/present")

unified_manifest.to_csv(f"{PROJECT}/data/manifest_unified.csv", index=False)
print("\nSaved to Drive: manifest_unified.csv")


Unified manifest shape: (6594, 5)

Per-source binary label counts:
source         label_binary
circor2022     0               2544
               1                509
physionet2016  0               2725
               1                816
Name: count, dtype: int64

Total unique groups (for split): 4618
Overall class balance: 20.12% abnormal/present

Saved to Drive: manifest_unified.csv


### Finding — Unified Manifest Summary
| Metric | Value |
|---|---|
| Total recordings | 6,594 |
| PhysioNet 2016 | 3,541 |
| CirCor 2022 (after dropping 'Unknown') | 3,053 |
| Label = 0 (normal/absent) | 5,269 (79.9%) |
| Label = 1 (abnormal/present) | 1,325 (20.1%) |
| Overall class ratio | ~4:1 (normal:abnormal) |
| Unique groups for splitting | 4,618 |

The `group_id` column is critical for preventing data leakage in Phase 6.
For CirCor, multiple recordings from the same patient share a `group_id`
(their patient ID prefix) — if windows from the same patient appeared in both
train and test, the model could partially "recognise" the patient rather than
genuinely generalise to new individuals.


## Step 7 — Listening Sanity Check

Before running any preprocessing, listen to a representative sample of clips
from each class. This builds intuition for what the model needs to learn and
reveals noise characteristics early.

**Playback note — sample rate issue:**
PhysioNet 2016 recordings are sampled at 2,000 Hz. Most browsers'
`<audio>` elements require at least ~8,000 Hz to render properly —
files at 2,000 Hz show `0:00/0:00` and refuse to play, even though
the file is completely valid (confirmed via `soundfile.info()` which reports
correct duration, format, and PCM encoding). Fix: upsample to 8,000 Hz
**for playback only** via `librosa.resample()`. The actual processing pipeline
always uses the native 2,000 Hz rate.


In [ ]:
import librosa
import IPython.display as ipd

LISTEN_SR = 8000  # playback only — all actual processing uses native rate

for label in [0, 1]:
    print(f"--- label_binary = {label} ({'normal/absent' if label==0 else 'abnormal/present'}) ---")
    sample = unified_manifest[unified_manifest['label_binary'] == label].sample(2, random_state=1)
    for _, row in sample.iterrows():
        print(f"  {row['source']} | {row['label_mapped']} | {os.path.basename(row['path'])}")
        y, sr = librosa.load(row['path'], sr=None)
        y_listen = librosa.resample(y, orig_sr=sr, target_sr=LISTEN_SR) if sr < LISTEN_SR else y
        ipd.display(ipd.Audio(y_listen, rate=LISTEN_SR if sr < LISTEN_SR else sr))


### Listening Findings
**Label = 0 (normal/absent):**
- Clear "lub-dub" rhythm audible at ~60–90 BPM
- Consistent silence between beats
- Some background hiss/rustling present even in "clean" clips
- All 4 clips required volume at 100% to hear clearly — expected for 
  low-frequency PCG content (20–400 Hz) through laptop speakers which roll off
  below ~100–150 Hz

**Label = 1 (abnormal/present):**
- "Tuck-tuck" pattern audible between the main beats in the CirCor clip
  — this is the murmur, a whooshing/blowing sound in the systolic phase
- One PhysioNet abnormal clip sounded similar to normal, suggesting a subtle
  or low-grade murmur (these become the hardest cases for the model)

**Key insight:** The datasets have a genuinely high noise floor even in
"normal" recordings. This is by design — PhysioNet 2016 was deliberately
collected in non-clinical, real-world environments. The preprocessing pipeline
(Phase 3) must handle this noise without destroying the heart sound content.


---
# PHASE 3 — Preprocessing Pipeline

## Design Goals
1. Standardise all clips to a single sample rate (2,000 Hz)
2. Remove noise outside the physiological PCG frequency range (20–400 Hz)
3. Eliminate rare spike artifacts without damaging real S1/S2 beats
4. Reduce broadband hiss while preserving murmur content
5. Remove leading/trailing silence
6. Normalise amplitude consistently across all clips

## Pipeline Order
**Resample → Bandpass filter → Despike → Wavelet denoise → Trim silence → Normalize**

The order matters: filtering before despiking avoids the bandpass filter
"spreading" a spike into a longer artifact; denoising after despiking avoids
the wavelet threshold being influenced by extreme values.


In [ ]:
!pip install -q pywavelets


## Step 8 — Define Preprocessing Functions


In [ ]:
import numpy as np
import librosa
import pywt
from scipy.signal import butter, filtfilt

TARGET_SR = 2000  # PhysioNet 2016 native; CirCor (4000 Hz) downsampled to match


def bandpass_filter(y, sr=TARGET_SR, low=20, high=400):
    """
    4th-order Butterworth bandpass filter: 20–400 Hz.
    Removes sub-bass rumble (below 20 Hz, e.g. body movement, DC drift)
    and high-frequency noise (above 400 Hz, e.g. electronics, cable friction).
    The 20–400 Hz band captures all clinically relevant PCG content:
    S1 (~40–160 Hz), S2 (~50–250 Hz), and murmurs (~25–400 Hz).
    Uses zero-phase filtfilt to avoid phase distortion at beat onsets.
    """
    nyq = sr / 2
    b, a = butter(4, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, y)


def despike(y, z_thresh=8):
    """
    Remove isolated artifact clicks using a MAD-based robust z-score.

    Why MAD (Median Absolute Deviation) instead of percentile clipping?
    A fixed percentile cut (e.g. clip everything above the 99.5th percentile)
    affects ALL clips equally regardless of their content. Since real S1/S2
    heartbeat peaks are themselves sharp and recur dozens of times per clip,
    a percentile-based approach inevitably clips legitimate beats — confirmed
    by listening: the resulting audio sounded 'flat', with beats not fully
    rising and falling.

    MAD is robust to the presence of many similar peaks (the real heartbeats)
    because median/MAD statistics are not pulled toward the outlier-dominated
    mean. Only samples that are extreme outliers relative to the clip's own
    distribution (z_thresh=8 standard deviations) are treated as spikes.
    Spike repair: replace the flagged sample with the median of its immediate
    neighbors (±2 samples) rather than zeroing or clipping.
    """
    med = np.median(y)
    mad = np.median(np.abs(y - med)) + 1e-8
    z = np.abs(y - med) / (1.4826 * mad)   # 1.4826 makes MAD ≈ std dev
    spike_idx = np.where(z > z_thresh)[0]

    y_clean = y.copy()
    for i in spike_idx:
        lo, hi = max(i - 2, 0), min(i + 3, len(y))
        neighbors = np.delete(y[lo:hi], np.where(np.arange(lo, hi) == i))
        if len(neighbors) > 0:
            y_clean[i] = np.median(neighbors)
    return y_clean


def wavelet_denoise(y, wavelet='db4', level=4):
    """
    Soft-threshold wavelet denoising using the Donoho-Johnstone universal
    threshold: σ * sqrt(2 * log(N)), where σ is estimated from the finest
    detail coefficients via the MAD estimator (robust to non-Gaussian noise).
    Daubechies 4 (db4) is chosen for its compact support and similarity to
    the smooth envelope shapes of S1/S2 waveforms.
    """
    coeffs = pywt.wavedec(y, wavelet, level=level)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(len(y)))
    coeffs[1:] = [pywt.threshold(c, threshold, mode='soft') for c in coeffs[1:]]
    return pywt.waverec(coeffs, wavelet)[:len(y)]


def trim_silence(y, top_db=25):
    """Remove leading/trailing near-silent segments (< 25 dB below peak)."""
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    return y_trimmed if len(y_trimmed) > 0 else y


def normalize_amplitude(y):
    """
    True-peak normalization: scale so max(|y|) = 1.0.

    Why true-peak (not percentile-based)?
    An earlier version used 99th-percentile normalization as a workaround for
    spike-dominated clips. This caused ~1% of samples (including real heartbeat
    peaks, which are legitimately the loudest events in the clip) to land above
    ±1.0 after scaling, causing hard clipping on playback and 'flattened'
    sounding beats. Since despike() already handles true outliers, true-peak
    normalization is safe and gives mathematically clean output (peak exactly 1.0).
    """
    y = y - np.mean(y)
    peak = np.max(np.abs(y))
    return y / peak if peak > 0 else y


def signal_quality_score(y):
    """
    Crest factor (peak/RMS) as a proxy for signal quality.
    High crest factor = signal dominated by isolated peaks (potential artifact).
    NOTE: This metric was evaluated and found to be unreliable as a standalone
    filter for this dataset (see Step 11). It is logged for analysis only.
    """
    rms = np.sqrt(np.mean(y ** 2))
    peak = np.max(np.abs(y))
    return peak / (rms + 1e-8)


def preprocess_clip(path):
    """
    Full preprocessing pipeline.
    Returns: (clean_signal: np.float32, sample_rate: int, quality_score: float)
    """
    y, sr = librosa.load(path, sr=None)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
    y = bandpass_filter(y, TARGET_SR)
    y = despike(y)
    quality = signal_quality_score(y)
    y = wavelet_denoise(y)
    y = trim_silence(y)
    y = normalize_amplitude(y)
    return y.astype(np.float32), TARGET_SR, quality


## Step 9 — Validate Pipeline on 3 Sample Clips

Always validate preprocessing on a small sample before running a batch job
across thousands of files. Two checks:
1. **Visual:** waveform should look less noisy/jagged after cleaning
2. **Auditory:** heart beats should still be clearly audible and not flattened
   or muffled; only background hiss should decrease

**Key check:** `peak` value printed after each clip should be exactly `1.0`
(confirming true-peak normalization ran correctly, not the old percentile version).


In [ ]:
import matplotlib.pyplot as plt
import IPython.display as ipd

test_rows = unified_manifest.sample(3, random_state=2)
for _, row in test_rows.iterrows():
    y_raw, sr_raw = librosa.load(row['path'], sr=None)
    y_clean, sr_clean, quality = preprocess_clip(row['path'])

    print(f"{row['label_mapped']} | quality score: {quality:.2f} | peak: {np.max(np.abs(y_clean)):.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].plot(y_raw)
    axes[0].set_title(f"Raw ({row['label_mapped']})")
    axes[0].set_xlabel("Samples"); axes[0].set_ylabel("Amplitude")
    axes[1].plot(y_clean)
    axes[1].set_title("Cleaned (after full pipeline)")
    axes[1].set_xlabel("Samples"); axes[1].set_ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

    y_clean_listen = librosa.resample(y_clean, orig_sr=sr_clean, target_sr=8000)
    ipd.display(ipd.Audio(y_clean_listen, rate=8000))


### Validation Findings
- **Peak = 1.0** confirmed for all 3 clips — true-peak normalization working
- **Waveform:** cleaned versions show visibly reduced background noise
  floor while beat peaks retain full amplitude
- **Audio:** "lub-dub" rhythm clearly audible in cleaned versions; background
  hiss reduced; beats not flattened or muffled

**Iteration history (documented for transparency):**
During development, three issues were caught in this validation step:
1. **Percentile despike caused beat flattening:** Clipping everything above the
   99.5th percentile removed real S1/S2 peaks. Fixed by switching to
   MAD-based z-score despike (threshold=8).
2. **Percentile normalization caused playback clipping:** Using 99th-percentile
   scaling left ~1% of samples above ±1.0, causing audible flattening.
   Fixed by reverting to true-peak normalization (safe after despiking).
3. **Stale function definition in Jupyter:** After fixing the normalization
   function, the old version remained in memory because the cell wasn't
   re-executed. Peak showed 2.3 instead of 1.0. Fixed by restarting runtime
   and re-running all cells top-to-bottom.


## Step 10 — Run Full Preprocessing Batch

Process all 6,293 clips (validation subfolder already excluded from
`unified_manifest`). Each cleaned signal is saved as a float32 NumPy array
(`.npy`) indexed by its manifest row number. Saving as `.npy` rather than
`.wav` avoids audio encoding overhead and is faster to load during training.


In [ ]:
from tqdm import tqdm

os.makedirs('/content/work/processed', exist_ok=True)
processed_paths, quality_scores, failed = [], [], []

for idx, row in tqdm(unified_manifest.iterrows(), total=len(unified_manifest)):
    try:
        y_clean, sr_clean, quality = preprocess_clip(row['path'])
        out_path = f"/content/work/processed/{idx}.npy"
        np.save(out_path, y_clean)
        processed_paths.append(out_path)
        quality_scores.append(quality)
    except Exception as e:
        processed_paths.append(None)
        quality_scores.append(None)
        failed.append((row['path'], str(e)))

unified_manifest['processed_path'] = processed_paths
unified_manifest['quality_score'] = quality_scores

print(f"Failed: {len(failed)} / {len(unified_manifest)}")
if failed:
    print("Failed files:", failed[:3])


100%|██████████| 6293/6293 [09:12<00:00, 11.39it/s]Failed: 0 / 6293


### Result
- **6,293 / 6,293 clips processed successfully** — 0 failures
- Runtime: ~9 minutes on Colab CPU (preprocessing is deterministic and single-threaded)
- Output: 6,293 `.npy` files at `/content/work/processed/`, indexed 0–6292


## Step 11 — Quality Score Distribution Analysis


In [ ]:
import matplotlib.pyplot as plt

unified_manifest['quality_score'].hist(bins=50, edgecolor='white')
plt.axvline(
    unified_manifest['quality_score'].quantile(0.95),
    color='red', linestyle='--', label='95th percentile'
)
plt.title("Quality Score (Crest Factor) Distribution — All 6,293 Clips")
plt.xlabel("Crest Factor (peak / RMS)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

print(unified_manifest['quality_score'].describe())


count    6293.000000
mean       16.812282
std        11.877692
min         2.890362
25%         8.091613
50%        11.873846
75%        22.546643
max       109.961106
Name: quality_score, dtype: float64


### Finding — Quality Score Distribution
- **Shape:** Right-skewed with a long tail (typical of impulsive noise metrics)
- **Median:** 11.87, **Mean:** 16.81 — the mean is pulled right by high-crest outliers
- **95th percentile:** ~42 — clips above this are candidates for artifact review

**Critical finding from spot-checking the top 5 highest-scoring clips:**
Crest factor scores ranged 76–110, yet the actual audio quality varied from
"clear but quiet" (needed max volume) down to "almost pure static." The same
numeric score described both good and bad recordings.

**Conclusion: crest factor is NOT a reliable standalone quality filter** for
this dataset. Real heartbeat peaks (S1/S2) are themselves the "impulsive"
events that drive up the crest factor — a clear, regular heartbeat and a
noisy, artifact-dominated clip can produce similar scores.

**Decision:** No clips are excluded based on quality score.
All 6,293 clips are retained for training for two reasons:
1. The model benefits from seeing realistic noise — this directly supports the
   "noise robustness" goal and the deployment scenario (non-clinical environments)
2. Signal quality gating is handled at **inference time** in the dashboard
   (Phase 10), not at training time — this is a more appropriate and flexible
   approach than pre-filtering the training set with an unreliable metric


## Step 12 — Save Final Manifest and Backup to Drive


In [ ]:
unified_manifest.to_csv(f"{PROJECT}/data/manifest_processed.csv", index=False)

# use cd-then-zip pattern to avoid nested path in the zip
!cd /content/work && zip -qr /content/processed_audio.zip processed/
!cp /content/processed_audio.zip "{PROJECT}/data/processed/"
!ls -lh "{PROJECT}/data/processed/"


total 1.1G
-rw------- 1 root root 1.1G processed_audio.zip


---
## Phases 2 & 3 Summary

### Phase 2 — Manifest Building
| Step | Action | Result |
|---|---|---|
| 3 | PhysioNet 2016 manifest | 3,541 recordings · 2,725 normal · 816 abnormal |
| 4–5 | CirCor 2022 per-recording expansion | 3,209 recordings (43 recovered by prefix glob) |
| 6 | Unified manifest | 6,594 total · 20.1% abnormal/present |
| 7 | Listening check | Murmurs audible in label=1 clips; high noise floor confirmed |

### Phase 3 — Preprocessing
| Step | Action | Result |
|---|---|---|
| 8 | Pipeline functions defined | 6 functions, all documented with design rationale |
| 9 | Validation on 3 clips | Peak = 1.0 ✅ · Audio quality improved ✅ |
| 10 | Full batch run | 6,293/6,293 success · 0 failures |
| 11 | Quality score analysis | Crest factor unreliable as filter — all clips retained |
| 12 | Backup to Drive | manifest_processed.csv + 1.1 GB processed_audio.zip |

### Key Design Decisions (for project report)
1. **MAD-based despiking** instead of percentile clipping — prevents
   removal of legitimate heartbeat peaks
2. **True-peak normalization** instead of percentile scaling — gives clean
   ±1.0 amplitude without clipping real peaks
3. **No quality-score-based filtering** — unreliable metric; noise robustness
   handled via augmentation at training time and gating at inference time
4. **Validation subfolder excluded** — not training data; including it would
   misrepresent the model's generalisability

### Next Step → Phase 4: Segmentation into Fixed-Length Windows
Each variable-length cleaned clip will be cut into 4-second windows with
50% overlap. This creates the fixed-size inputs the model requires and
significantly expands the training set through data augmentation by overlap.
